# INQUIRE_CSI Part 2a – Validation

> **Warning: this notebook does not run stand-alone.**
> It is an optional part of `INQUIRE_CSI_Pt2_Analysis_and_Validation.ipynb` and is run from there when `RUN_VALIDATION = True` (cell *Run validation (optional)*). It uses variables, functions and settings defined earlier in that notebook (e.g., `source_cpds`, `build_problem`, `greedy_cover`, `best_alpha`, `VALIDATION_SEED`) and runs in the same kernel.
> To run the validation, open the Part 2 notebook and set `RUN_VALIDATION = True`; to skip it, set `RUN_VALIDATION = False`.

In [ ]:
# Guard: stop if this notebook is run on its own
if "source_cpds" not in globals() or "VALIDATION_SEED" not in globals() or "best_alpha" not in globals():
    raise RuntimeError(
        "INQUIRE_CSI_Pt2a_Validation does not run stand-alone. "
        "Run it from INQUIRE_CSI_Pt2_Analysis_and_Validation.ipynb (RUN_VALIDATION = True)."
    )


In [ ]:
# Synthetic perturbation validation
majorsep()
logging.info(f"Running synthetic robustness validation (dropout & contamination) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

rng = np.random.default_rng(VALIDATION_SEED)

perturb_results = []

# Perturbation strengths
dropout_rates = [0.05, 0.1, 0.2]
contam_rates = [0.0, 0.05, 0.1]

source_ids_all = list(source_cpds.keys())

def perturb_sources(true_sources, dropout=0.1, contam=0.05):
    perturbed = set()
    all_compounds = sorted({c for s in source_cpds.values() for c in s}) # sorted → reproducible between kernel restarts

    for sid in true_sources:
        comps = sorted(source_cpds[sid]) # sorted → reproducible between kernel restarts

        # dropout
        kept = {c for c in comps if rng.random() > dropout}

        # contamination
        n_contam = int(contam * len(comps))
        contam_added = set(rng.choice(all_compounds, size=n_contam, replace=False))
        perturbed |= kept | contam_added

    return perturbed

outer_iter = [(d, c) for d in dropout_rates for c in contam_rates]

for dropout, contam in tqdm(outer_iter, desc="Dropout × contamination"):
    for rep in tqdm(range(100), desc=f"rep (d={dropout}, c={contam})", leave=False):
        true_sources = rng.choice(source_ids_all, size=50, replace=False)

        true_compounds = set()
        for sid in true_sources:
            true_compounds |= source_cpds[sid]

        U = sorted(true_compounds)
        U_set = set(U)

        perturbed_U = perturb_sources(true_sources, dropout, contam)

        A, source_index, P = build_problem(perturbed_U, household_id=None)
        if A is None:
            continue
            
        x, _ = greedy_cover(A, list(perturbed_U), P, source_index, alpha=best_alpha)

        recovered = {source_index[i] for i in np.where(x == 1)[0]}
        false_sources = recovered - set(true_sources)
        true_sources_set = set(true_sources)
        
        source_precision = (
            len(recovered & true_sources_set) / len(recovered)
            if recovered else 0
        )
        
        false_discovery_rate = (
            len(false_sources) / len(recovered)
            if recovered else 0
        )

        recovered_compounds = set()
        for sid in recovered:
            recovered_compounds |= source_cpds[sid]

        precision = (
            len(recovered_compounds & true_compounds)
            / len(recovered_compounds)
            if recovered_compounds else 0
        )

        recall = (
            len(recovered_compounds & true_compounds)
            / len(true_compounds)
            if true_compounds else 0
        )

        perturb_results.append({
            "dropout": dropout,
            "contamination": contam,
            "precision": precision,
            "recall": recall,
            "source_precision": source_precision,
            "false_discovery_rate": false_discovery_rate
        })

perturb_df = pd.DataFrame(perturb_results)

summary_perturb = perturb_df.groupby(["dropout", "contamination"]).mean()
minorsep()
display(summary_perturb.round(3))

minorsep()
logging.info(f"Completed synthetic perturbation validation (dropout & contamination) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation with overlap sensitivity
majorsep()
logging.info(f"Running overlap-sensitivity synthetic validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

validation_results = []

rng = np.random.default_rng(VALIDATION_SEED)

source_ids_all = list(source_cpds.keys())
mixture_sizes = [10, 100, 200, 300]
replicates = 100
overlap_thresholds = [0.10, 0.50, "random"]

def choose_sources_overlap_control(n_sources, max_jaccard=None):
    candidates = list(source_cpds.keys())
    rng.shuffle(candidates)
    selected = []

    for sid in candidates:
        if len(selected) == 0:
            selected.append(sid)
            continue

        if max_jaccard is None:
            # random mode: no overlap constraint at all
            selected.append(sid)
        else:
            current = source_cpds[sid]
        
            ok = True
        
            for chosen in selected:
                other = source_cpds[chosen]
                inter = len(current & other)
                union = len(current | other)
                j = inter / union if union else 0.0
        
                if j > max_jaccard:
                    ok = False
                    break
        
            if ok:
                selected.append(sid)

        if len(selected) == n_sources:
            break

    return selected


outer = [(o, n) for o in overlap_thresholds for n in mixture_sizes]

for overlap, n_true in tqdm(outer, desc="Overlap × mixture size"):
    for rep in tqdm(range(replicates), desc=f"rep (o={overlap}, n={n_true})", leave=False):
        max_jaccard = None if overlap == "random" else float(overlap)
        true_sources = choose_sources_overlap_control(n_true, max_jaccard)

        # record when the overlap constraint could not be met and a random mixture was used instead
        fallback_used = len(true_sources) < n_true
        if fallback_used:
            true_sources = rng.choice(source_ids_all, size=n_true, replace=False)
        
        true_compounds = set()
        
        for sid in true_sources:
            true_compounds |= source_cpds[sid]

        U = list(true_compounds)

        A, source_index, P = build_problem(true_compounds, household_id=None)
        
        if A is None:
            continue
        
        x, _ = greedy_cover(A, U, P, source_index, alpha=best_alpha)

        recovered = {source_index[i] for i in np.where(x == 1)[0]}
        true_sources = set(true_sources)

        tp = len(recovered & true_sources)
        fp = len(recovered - true_sources)
        
        source_precision = (tp / len(recovered) if recovered else 0)
        source_recall = (tp / len(true_sources) if true_sources else 0)
        
        source_f1 = (
            2 * source_precision * source_recall /
            (source_precision + source_recall)
            if (source_precision + source_recall) > 0 else 0
        )
        
        false_discovery_rate = (
            fp / len(recovered)
            if recovered else 0
        )

        recovered_compounds = set()
        
        for sid in recovered:
            recovered_compounds |= source_cpds[sid]

        precision = (
            len(recovered_compounds & true_compounds)
            / len(recovered_compounds)
            if recovered_compounds else 0
        )

        validation_results.append({
            "overlap": overlap,
            "fallback_random": fallback_used, # mean of this column = fraction of random fallback mixtures
            "n_true_sources": n_true,
            "precision": precision,
            "source_precision": source_precision,
            "source_recall": source_recall,
            "source_f1": source_f1,
            "false_discovery_rate": false_discovery_rate,
            "n_selected_sources": len(recovered),
            "tp_sources": tp,
            "fp_sources": fp
        })

overlap_validation_df = pd.DataFrame(validation_results)
overlap_summary = overlap_validation_df.groupby(["overlap", "n_true_sources"]).mean()

minorsep()
logging.info(f"Random fallback mixtures (overlap constraint not met): {int(overlap_validation_df['fallback_random'].sum()):,} of {len(overlap_validation_df):,} replicates")
logging.info(f"Rows with fallback_random > 0 in the table below are (partly) random mixtures, not overlap-controlled ones")

minorsep()
display(overlap_summary.round(3))
minorsep()

logging.info(f"Completed overlap sensitivity analysis | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation (chemistry-based)
majorsep()
logging.info(f"Running synthetic recovery validation (chemistry-based) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

validation_results = []

rng = np.random.default_rng(VALIDATION_SEED)
source_ids_all = list(source_cpds.keys())
mixture_sizes = [10, 100, 200, 300]
replicates = 100

def choose_distinct_sources(source_cpds, n_sources, rng, max_jaccard=0.25):
    candidates = list(source_cpds.keys())
    rng.shuffle(candidates)

    selected = []

    for sid in candidates:
        current = source_cpds[sid]
        keep = True

        for chosen in selected:
            other = source_cpds[chosen]
            inter = len(current & other)
            union = len(current | other)
            j = inter / union if union else 0

            if j > max_jaccard:
                keep = False
                break

        if keep:
            selected.append(sid)

        if len(selected) == n_sources:
            break

    return selected

for n_true in mixture_sizes:
    for rep in tqdm(range(replicates), desc=f"Synthetic mixtures ({n_true} sources & {replicates} replicates)"):
        true_sources = choose_distinct_sources(source_cpds, n_true, rng, max_jaccard=0.25)

        # record when the Jaccard constraint could not be met and a random mixture was used instead
        fallback_used = len(true_sources) < n_true
        if fallback_used:
            true_sources = rng.choice(source_ids_all, size=n_true, replace=False)

        true_sources_set = set(true_sources)

        true_compounds = set()
        for sid in true_sources:
            true_compounds.update(source_cpds[sid])

        U = sorted(true_compounds)

        if len(U) == 0:
            continue

        U_set = set(U)

        A, source_index, P = build_problem(U_set, household_id=None)
        if A is None:
            continue
        
        x, e = greedy_cover(A, U, P, source_index, alpha=best_alpha)

        recovered_sources = {source_index[i] for i in np.where(x == 1)[0]}
        recovered_sources_set = set(recovered_sources)
        recovered_compounds = set()
        
        for sid in recovered_sources:
            recovered_compounds.update(source_cpds[sid])

        # Compound-level metrics
        intersection = len(true_compounds & recovered_compounds)
        union = len(true_compounds | recovered_compounds)
        compound_precision = (intersection / len(recovered_compounds) if recovered_compounds else 0)
        compound_recall = (intersection / len(true_compounds) if true_compounds else 0)

        compound_f1 = (
            2 * compound_precision * compound_recall /
            (compound_precision + compound_recall)
            if (compound_precision + compound_recall) > 0 else 0
        )

        jaccard = intersection / union if union else 0

        expansion_factor = (
            len(recovered_compounds) / len(true_compounds)
            if true_compounds else np.nan
        )

        # Source-level metrics
        source_precision = (
            len(recovered_sources_set & true_sources_set)
            / len(recovered_sources_set)
            if recovered_sources_set else 0
        )

        source_recall = (
            len(recovered_sources_set & true_sources_set)
            / len(true_sources_set)
            if true_sources_set else 0
        )

        false_discovery_rate = (
            len(recovered_sources_set - true_sources_set)
            / len(recovered_sources_set)
            if recovered_sources_set else 0
        )

        validation_results.append({
            "n_true_sources": n_true,
            "fallback_random": fallback_used,
            # compound-level
            "compound_precision": compound_precision,
            "compound_recall": compound_recall,
            "compound_f1": compound_f1,
            "jaccard": jaccard,
            "compound_expansion_factor": expansion_factor,
            # source-level
            "source_precision": source_precision,
            "source_recall": source_recall,
            "false_discovery_rate": false_discovery_rate,
            # diagnostics
            "n_true_compounds": len(true_compounds),
            "n_recovered_compounds": len(recovered_compounds),
            "n_recovered_sources": len(recovered_sources_set),
        })

synth_validation_df = pd.DataFrame(validation_results)

synth_summary = (
    synth_validation_df
    .groupby("n_true_sources")
    .agg(
        compound_precision=("compound_precision", "mean"),
        compound_recall=("compound_recall", "mean"),
        compound_f1=("compound_f1", "mean"),
        jaccard=("jaccard", "mean"),
        compound_expansion_factor=("compound_expansion_factor", "mean"),
        source_precision=("source_precision", "mean"),
        source_recall=("source_recall", "mean"),
        false_discovery_rate=("false_discovery_rate", "mean"),
        fallback_random_rate=("fallback_random", "mean"), # fraction of random fallback mixtures
        mean_true_compounds=("n_true_compounds", "mean"),
        mean_recovered_compounds=("n_recovered_compounds", "mean"),
        mean_recovered_sources=("n_recovered_sources", "mean"),
    )
)

minorsep()
logging.info(f"Synthetic validation summary:")
display(synth_summary.round(3))

minorsep()
logging.info(f"Done running synthetic recovery validation (chemistry-based) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation (chemistry-based) summary statistics
synth_summary_2 = (
    synth_validation_df
    .groupby("n_true_sources")
    .agg(
        mean_precision=("compound_precision", "mean"),
        sd_precision=("compound_precision", "std"),
        n=("compound_precision", "count"),
        mean_recovered_sources=("n_recovered_sources", "mean")
    )
)

# Standard error
synth_summary_2["se_precision"] = (synth_summary_2["sd_precision"] / np.sqrt(synth_summary_2["n"]))

# 95% confidence interval
synth_summary_2["precision_ci_low"] = (synth_summary_2["mean_precision"] - 1.96 * synth_summary_2["se_precision"])
synth_summary_2["precision_ci_high"] = (synth_summary_2["mean_precision"] + 1.96 * synth_summary_2["se_precision"])

# Keep values within [0,1]
synth_summary_2["precision_ci_low"] = synth_summary_2["precision_ci_low"].clip(lower=0)
synth_summary_2["precision_ci_high"] = synth_summary_2["precision_ci_high"].clip(upper=1)

majorsep()
logging.info("Synthetic validation summary:")
minorsep()
display(synth_summary_2.round(3))
majorsep()


In [ ]:
# Synthetic recovery validation plot
majorsep()
logging.info(f"Plotting synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Summary statistics for plotting
plot_df = synth_summary_2.reset_index()

# Error bars
lower_err = (plot_df["mean_precision"] - plot_df["precision_ci_low"])
upper_err = (plot_df["precision_ci_high"] - plot_df["mean_precision"])

fig, axes = plt.subplots(1, 2, figsize=(7, 3), constrained_layout=True)

# Left panel
axes[0].errorbar(
    plot_df["n_true_sources"],
    plot_df["mean_precision"],
    yerr=[lower_err, upper_err],
    fmt="o-",
    linewidth=2,
    capsize=5,
    markersize=7
)

axes[0].set_xlabel("True sources in synthetic mixture")
axes[0].set_ylabel("Compound precision")
axes[0].set_ylim(0, 1.02)
axes[0].set_xticks(plot_df["n_true_sources"])
axes[0].grid(alpha=0.3)
axes[0].set_title("A", loc="left", fontweight="bold")

# Right panel
axes[1].plot(
    plot_df["n_true_sources"],
    plot_df["mean_recovered_sources"],
    "o-",
    linewidth=2,
    markersize=7
)

axes[1].set_xlabel("True sources in synthetic mixture")
axes[1].set_ylabel("Mean recovered sources")
axes[1].set_xticks(plot_df["n_true_sources"])
axes[1].grid(alpha=0.3)
axes[1].set_title("B", loc="left", fontweight="bold")

plt.savefig(VAL_DIR / "synthetic_validation_precision.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation results plot
majorsep()
logging.info(f"Plotting synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

mixture_sizes = sorted(synth_validation_df["n_true_sources"].unique())

data = [
    synth_validation_df.loc[
        synth_validation_df["n_true_sources"] == n,
        "compound_precision"
    ].values
    for n in mixture_sizes
]

plt.figure(figsize=(5, 3))
plt.boxplot(data, tick_labels=mixture_sizes, showfliers=False)
plt.xlabel("True sources in synthetic mixture")
plt.ylabel("Compound precision")
plt.ylim(0, 1.02)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(VAL_DIR / "synthetic_validation_boxplot.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Cost-consistent validation
majorsep()
logging.info(f"Running cost-consistent synthetic validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

cost_validation = []

rng = np.random.default_rng(VALIDATION_SEED)

mixture_sizes = [10, 100, 200, 300]
replicates = 100

source_ids_all = list(source_cpds.keys())

# True sources are sampled proportionally to their household-independent prior,
# i.e. sources the model considers more plausible are drawn more often
prior_weights = np.array([source_prior_lookup[s] for s in source_ids_all], dtype=float)
weights = prior_weights / prior_weights.sum()

for n_true in mixture_sizes:
    for rep in tqdm(range(replicates), desc=f"Cost-consistent (mixture size {n_true})"):
        true_sources = rng.choice(
            source_ids_all,
            size=n_true,
            replace=False,
            p=weights
        )

        true_sources = list(true_sources)
        true_sources_set = set(true_sources)
        true_compounds = set()

        for sid in true_sources:
            true_compounds |= source_cpds[sid]

        U = sorted(true_compounds)

        A, source_index, P = build_problem(true_compounds, household_id=None)
        if A is None:
            continue

        # compare costs on the same objective the optimiser minimises (prior/coverage cost + alpha * penalty)
        full_cost = calculate_source_costs(source_index, set(U), P, best_alpha)
        cost_lookup = dict(zip(source_index, full_cost))

        x, _ = greedy_cover(A, U, P, source_index, alpha=best_alpha)

        recovered_sources = {source_index[i] for i in np.where(x == 1)[0]}
        recovered_compounds = set()

        for sid in recovered_sources:
            recovered_compounds |= source_cpds[sid]

        intersection = len(recovered_compounds & true_compounds)
        compound_precision = (intersection / len(recovered_compounds) if recovered_compounds else 0)
        compound_recall = (intersection / len(true_compounds) if true_compounds else 0)

        source_precision = (len(recovered_sources &
                true_sources_set
            ) / len(recovered_sources)
            if recovered_sources else 0)

        source_recall = (len(recovered_sources &
                true_sources_set
            ) / len(true_sources_set))

        recovered_cost = sum(cost_lookup[s] for s in recovered_sources)
        true_cost = np.sum([cost_lookup[s] for s in true_sources if s in cost_lookup])

        cost_validation.append({
            "n_true_sources": n_true,
            "compound_precision": compound_precision,
            "compound_recall": compound_recall,
            "source_precision": source_precision,
            "source_recall": source_recall,
            "cost_ratio": (recovered_cost / true_cost if true_cost > 0 else np.nan)})

cost_df = pd.DataFrame(cost_validation)
display(cost_df.groupby("n_true_sources").mean().round(3))

minorsep()
logging.info(f"Completed cost-consistent validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Noise robustness validation
majorsep()
logging.info(f"Running noise robustness validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

noise_results = []
missing_rates = [0.1, 0.3, 0.5]
mixture_sizes = [10, 100, 200, 300]
replicates = 100

for n_true_sources in mixture_sizes:
    for missing_rate in missing_rates:
        for rep in range(replicates):
            if rep % 50 == 0:
                tqdm.write(f"{n_true_sources} sources | missing {missing_rate * 100}% | replicates {rep}")
            
            # Generate synthetic mixture
            true_sources = rng.choice(source_ids_all, size=n_true_sources, replace=False)
            true_union = set()

            for sid in true_sources:
                true_union.update(source_cpds[sid])

            if len(true_union) < 10:
                continue

            true_union = set(true_union)

            # Remove compounds (simulate missing detections)
            n_remove = int(len(true_union) * missing_rate)

            if n_remove > 0:
                removed_idx = rng.choice(
                    len(true_union),
                    size=n_remove,
                    replace=False
                )

                removed_idx_set = set(removed_idx.tolist())
                observed = {
                    c for i, c in enumerate(sorted(true_union))
                    if i not in removed_idx_set
                } # sorted → reproducible between kernel restarts
            else:
                observed = true_union.copy()

            U_set = set(observed)

            A, source_index, P = build_problem(U_set, household_id=None)
            if A is None:
                continue
            
            U = sorted(observed)
            x, e = greedy_cover(A, U, P, source_index, alpha=best_alpha)

            recovered_sources = {
                source_index[i]
                for i in np.where(x == 1)[0]
            }

            # Compare compound space
            recovered_union = set()

            for sid in recovered_sources:
                recovered_union.update(source_cpds[sid])

            true_set = set(true_union) # ensure consistent casting
            intersection = len(set(true_union) & recovered_union)
            union = len(set(true_union) | recovered_union)

            jaccard = (len(set(true_union) & recovered_union) / union if union > 0 else 0)

            # Source-level metrics
            true_sources_set = set(true_sources)
            tp = len(recovered_sources & true_sources_set)
            fp = len(recovered_sources - true_sources_set)

            precision = (tp / len(recovered_sources) if len(recovered_sources) > 0 else 0)
            recall = tp / len(true_sources) if len(true_sources) > 0 else 0

            f1 = (
                2 * precision * recall / (precision + recall)
                if (precision + recall) > 0
                else 0
            )

            noise_results.append({
                "n_true_sources": n_true_sources,
                "missing_rate": missing_rate,
                "jaccard": jaccard,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "false_discovery_rate": (
                    fp / len(recovered_sources)
                    if recovered_sources else 0
                ),
                "n_recovered_sources": len(recovered_sources)
            })

noise_df = pd.DataFrame(noise_results)

minorsep()
logging.info(f"Noise robustness validation summary:")

noise_summary = (
    noise_df
    .groupby(["n_true_sources", "missing_rate"])
    .agg(
        mean_jaccard=("jaccard", "mean"),
        median_jaccard=("jaccard", "median"),
        std_jaccard=("jaccard", "std"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        mean_false_discovery_rate=("false_discovery_rate", "mean"),
        mean_recovered_sources=("n_recovered_sources", "mean")
    )
)

display(noise_summary)

minorsep()
logging.info(f"Done running noise robustness validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# GREEDY stability validation
majorsep()
logging.info(f"Running GREEDY stability validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

greedy_stability_results = []
rng = np.random.default_rng(VALIDATION_SEED)

drop_fraction = 0.05
mixture_sizes = [10, 100, 200, 300]
replicates = 100

for n_true in mixture_sizes:
    for rep in tqdm(range(replicates), desc=f"GREEDY stability (mixture size {n_true} & drop {drop_fraction * 100}%)"):
        true_sources = rng.choice(source_ids_all, size=n_true, replace=False)
        compounds = set()

        for sid in true_sources:
            compounds |= source_cpds[sid]

        U = sorted(compounds)

        A, source_index, P = build_problem(compounds, household_id=None)
        if A is None:
            continue
        
        x,_ = greedy_cover(A, U, P, source_index, alpha=best_alpha)

        baseline = {source_index[i] for i in np.where(x == 1)[0]}
        baseline_compounds = set()

        for sid in baseline:
            baseline_compounds |= source_cpds[sid]

        perturbed = set(compounds)

        remove = rng.choice(
            sorted(perturbed), # sorted → reproducible between kernel restarts
            size=int(drop_fraction * len(perturbed)),
            replace=False
        )

        perturbed -= set(remove)
        U2 = sorted(perturbed)
        A2 = []
        source_index2 = []

        for sid,cpds in source_cpds.items():
            elems=cpds & perturbed

            if elems:
                A2.append(elems)
                source_index2.append(sid)

        if len(A2) == 0:
            continue

        P2 = make_costfun(A2, source_index2, source_meta_lookup)
        x2, _ = greedy_cover(A2, U2, P2, source_index2, alpha=best_alpha)
        perturbed_solution = {source_index2[i] for i in np.where(x2==1)[0]}

        # Reconstructed compound space from perturbed solution
        perturbed_compounds = set()
        
        for sid in perturbed_solution:
            perturbed_compounds |= source_cpds[sid]

        compound_union = baseline_compounds | perturbed_compounds

        compound_solution_jaccard = (
            len(baseline_compounds & perturbed_compounds)
            / len(compound_union)
            if len(compound_union) > 0
            else 1.0
        )

        union = baseline | perturbed_solution

        greedy_stability = (
            len(baseline & perturbed_solution)
            / len(union)
            if len(union) > 0
            else 1.0
        )

        greedy_stability_results.append({
            "n_true_sources": n_true,
            "solution_jaccard": greedy_stability,
            "baseline_size": len(baseline),
            "perturbed_size": len(perturbed_solution),
            "compound_solution_jaccard": compound_solution_jaccard
        })

greedy_stability_df = pd.DataFrame(greedy_stability_results)
minorsep()
display(greedy_stability_df.groupby("n_true_sources").mean().round(3))

minorsep()
logging.info(f"Completed GREEDY stability validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Beam Search posterior diagnostics
majorsep()
logging.info(f"Defining beam search posterior diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

def posterior_diagnostics(covers):
    if len(covers) == 0:
        return {
            "entropy": np.nan,
            "ess": np.nan,
            "top1": np.nan,
            "top5": np.nan
        }

    costs = np.array([c[2] for c in covers], dtype=float)
    costs -= costs.min()
    posterior = np.exp(-costs)
    posterior /= posterior.sum()
    entropy = -np.sum(posterior * np.log(posterior + 1e-300))
    ess = 1.0 / np.sum(posterior ** 2)
    top1 = posterior.max()
    top5 = np.sort(posterior)[::-1][:5].sum()

    return {
        "entropy": entropy,
        "ess": ess,
        "top1": top1,
        "top5": top5
    }

logging.info(f"Done defining beam search posterior diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Beam Search validation
majorsep()
logging.info(f"Running Beam Search validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

beam_widths = [5, 10, 30, 50]
beam_validation = []

rng = np.random.default_rng(VALIDATION_SEED)
source_ids_all = list(source_cpds.keys())

for beam_width in tqdm(beam_widths, desc="Beam widths"):
    runtimes = []
    cover_counts = []
    cover_costs = []
    entropies = []
    ess_values = []
    top1_values = []
    top5_values = []

    for rep in range(20):
        true_sources = rng.choice(source_ids_all, size=50, replace=False)
        U = set().union(*(source_cpds[sid] for sid in true_sources))
        A, source_index, P = build_problem(U, household_id=None, uncertainty_sigma=0.0)

        if A is None:
            continue

        t0 = time.perf_counter()

        covers = beam_search_covers(
            A,
            list(U),
            P,
            source_index,
            alpha=best_alpha_beam,
            beam_width=beam_width,
            max_covers=20
        )

        runtime = time.perf_counter() - t0
        runtimes.append(runtime)
        cover_counts.append(len(covers))

        if covers:
            cover_costs.append(np.mean([c[2] for c in covers]))

            diag = posterior_diagnostics(covers)
            entropies.append(diag["entropy"])
            ess_values.append(diag["ess"])
            top1_values.append(diag["top1"])
            top5_values.append(diag["top5"])
        
    beam_validation.append({
        "beam_width": beam_width,
        "runtime_sec": np.mean(runtimes),
        "covers_found": np.mean(cover_counts),
        "mean_cover_cost": np.mean(cover_costs),
        "posterior_entropy": np.mean(entropies),
        "effective_sample_size": np.mean(ess_values),
        "top1_posterior": np.mean(top1_values),
        "top5_posterior_mass": np.mean(top5_values)
    })

beam_validation_df = pd.DataFrame(beam_validation)
minorsep()
display(beam_validation_df)

minorsep()
logging.info(f"Done Beam Search validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Database redundancy (leave-one-source-out (LOSO))
majorsep()
logging.info(f"Running database redundancy (LOSO) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

loso_results = []

for sid_true in tqdm(source_ids_all, desc="LOSO validation"):
    U = source_cpds[sid_true]

    if len(U) < 5:
        continue

    database_minus_one = {sid: cpds for sid, cpds in source_cpds.items() if sid != sid_true}
    best_jaccard = 0
    best_match = None
    best_containment = 0 # moved here (was reset inside the loop below, so it was almost always 0)

    for sid_test, cpds in database_minus_one.items():
        j = (len(U & cpds) / len(U | cpds))
        
        containment = len(U & cpds) / len(U)
        # best_containment = 0

        if j > best_jaccard:
            best_jaccard = j
            best_containment = containment
            best_match = sid_test

    best_overlap = len(U & database_minus_one[best_match]) if best_match is not None else 0
    
    loso_results.append({
        "source_id": sid_true,
        "best_match": best_match,
        "best_jaccard": best_jaccard,
        "containment": best_containment,
        "n_compounds": len(U),
        "shared_compounds": best_overlap
    })

loso_df = pd.DataFrame(loso_results)
minorsep()
display(loso_df)

minorsep()
logging.info(f"Database redundancy (LOSO) summary:\n")
logging.info(f"Best Jaccard:")
logging.info(loso_df["best_jaccard"].describe())
minorsep()
logging.info(f"Worst Jaccard:")
display(loso_df.nsmallest(10, "best_jaccard"))
minorsep()
logging.info(f"Containment:")
logging.info(loso_df["containment"].describe())
minorsep()
logging.info(f"Worst containment:")
display(loso_df.sort_values("containment").head(10))

minorsep()
logging.info(f"Done running database redundancy (LOSO) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Plot database redundancy LOSO Jaccard values
majorsep()
logging.info(f"Plotting LOSO Jaccard values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

plt.figure(figsize=(5, 3))
loso_df["best_jaccard"].hist(bins=50)

plt.xlabel("LOSO Jaccard similarity")
plt.ylabel("Frequency")

plt.savefig(VAL_DIR / "loso_jaccard_plot.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted LOSO Jaccard values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# LOSO reconstruction validation (algorithm robustness)
majorsep()
logging.info(f"Running LOSO reconstruction validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

loso_reconstruction_results = []

MIN_COMPOUNDS = 5

eligible_sources = [sid for sid, cpds in source_cpds.items() if len(cpds) >= MIN_COMPOUNDS]

for sid_true in tqdm(eligible_sources, desc="LOSO reconstruction"):
    true_compounds = set(source_cpds[sid_true])

    # Remove true source from candidate database
    database_minus_one = {sid: cpds for sid, cpds in source_cpds.items() if sid != sid_true}

    U_set = true_compounds
    A = []
    source_index = []

    for sid, compounds in database_minus_one.items():
        elems = compounds & U_set
        if elems:
            A.append(elems)
            source_index.append(sid)

    if not A:
        continue
 
    P = make_costfun(A=A, source_index=source_index, source_meta_lookup=source_meta_lookup, household_id=None)

    # Recoverable chemistry only
    union_A = set().union(*A)
    recoverable_compounds = true_compounds & union_A

    if len(recoverable_compounds) < MIN_COMPOUNDS:
        continue

    recoverable_U = sorted(recoverable_compounds)

    x, _ = greedy_cover(A, recoverable_U, P, source_index, alpha=best_alpha)

    recovered_sources = {source_index[i] for i in np.where(x == 1)[0]}
    recovered_compounds = set()

    for sid in recovered_sources:
        recovered_compounds.update(source_cpds[sid])

    # The full chemistry of the selected substitute sources is compared with the true source:
    # - precision: fraction of compounds in the substitute sources that also belong to the true source
    # - recall: fraction of the true source's compounds explained by the substitutes
    intersection = len(true_compounds & recovered_compounds)

    compound_precision = (
        intersection / len(recovered_compounds)
        if recovered_compounds else 0
    )

    compound_recall = (
        intersection / len(true_compounds)
        if true_compounds else 0
    )

    compound_f1 = (
        2 * compound_precision * compound_recall /
        (compound_precision + compound_recall)
        if (compound_precision + compound_recall) > 0
        else 0
    )

    jaccard = (
        intersection / len(true_compounds | recovered_compounds)
        if len(true_compounds | recovered_compounds) > 0
        else 0
    )

    expansion_factor = (
        len(recovered_compounds) / len(true_compounds)
        if true_compounds else np.nan
    )

    loso_reconstruction_results.append({
        "source_id": sid_true,
        "compound_precision": compound_precision,
        "compound_recall": compound_recall,
        "compound_f1": compound_f1,
        "jaccard": jaccard,
        "expansion_factor": expansion_factor,
        "n_true_compounds": len(true_compounds),
        "n_recoverable_compounds": len(recoverable_compounds),
        "n_recovered_compounds": len(recovered_compounds),
        "recoverable_fraction": len(recoverable_compounds) / len(true_compounds),
        "n_selected_sources": len(recovered_sources),
        "selected_sources": sorted(recovered_sources)
    })

loso_reconstruction_df = pd.DataFrame(loso_reconstruction_results)

summary_loso = (
    loso_reconstruction_df[
        [
            "compound_precision",
            "compound_recall",
            "compound_f1",
            "jaccard",
            "expansion_factor",
            "recoverable_fraction",
            "n_selected_sources",
            "n_true_compounds",
            "n_recoverable_compounds",
            "n_recovered_compounds",
        ]
    ]
    .mean()
    .rename({
        "n_selected_sources": "mean_selected_sources",
        "n_true_compounds": "mean_true_compounds",
        "n_recoverable_compounds": "mean_recoverable_compounds",
        "n_recovered_compounds": "mean_recovered_compounds",
    })
    .to_frame(name="Mean")
)

minorsep()
logging.info(f"LOSO reconstruction summary:")
display(summary_loso.round(3))

minorsep()
logging.info(f"LOSO reconstruction distribution:")
display(
    loso_reconstruction_df[
        [
            "compound_precision",
            "compound_recall",
            "compound_f1",
            "jaccard",
            "expansion_factor",
            "recoverable_fraction",
        ]
    ].describe().round(3)
)

minorsep()
logging.info(f"Worst reconstructed sources:")
display(
    loso_reconstruction_df.sort_values("compound_f1").head(10)
)

minorsep()
logging.info(f"Best reconstructed sources:")
display(
    loso_reconstruction_df.sort_values(
        "compound_f1",
        ascending=False
    ).head(10)
)

minorsep()
logging.info(f"Done running LOSO reconstruction validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Validation interpretation engine
majorsep()
logging.info(f"Defining validation interpretation engine | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

METRIC_INFO = {
# Recovery
"compound_precision": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Fraction of recovered compounds that truly belong to the target sample.",
    "excellent_text": "Recovered compounds are highly specific with very few false positives.",
    "good_text": "Most recovered compounds are correct with limited contamination.",
    "moderate_text": "Recovery is informative but includes noticeable false positives.",
    "weak_text": "Recovered compounds contain many incorrect assignments."
},

"compound_recall": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Fraction of true compounds successfully recovered.",
    "excellent_text": "Nearly all true compounds are recovered.",
    "good_text": "Most true compounds are recovered.",
    "moderate_text": "Only part of the chemistry is recovered.",
    "weak_text": "Large fractions of true chemistry are missed."
},

"compound_f1": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Balanced measure combining compound precision and recall.",
    "excellent_text": "Recovery is simultaneously accurate and complete.",
    "good_text": "Recovery is consistently strong.",
    "moderate_text": "Recovery is acceptable but imperfect.",
    "weak_text": "Recovery quality is poor."
},

"source_precision": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Fraction of inferred sources that are truly correct.",
    "excellent_text": "Few incorrect products are proposed.",
    "good_text": "Most inferred products are correct.",
    "moderate_text": "Recovered sources include several false positives.",
    "weak_text": "Many inferred sources are incorrect."
},

"source_recall": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Fraction of true sources successfully identified.",
    "excellent_text": "Nearly all generating sources are recovered.",
    "good_text": "Most generating sources are recovered.",
    "moderate_text": "Only part of the generating sources are recovered.",
    "weak_text": "Most generating sources remain unidentified."
},

"source_f1": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Balanced source-level recovery.",
    "excellent_text": "Excellent overall source identification.",
    "good_text": "Reliable source identification.",
    "moderate_text": "Moderately accurate source recovery.",
    "weak_text": "Poor source recovery."
},

"jaccard": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Similarity between recovered and true compound sets.",
    "excellent_text": "Recovered chemistry almost perfectly matches the truth.",
    "good_text": "Recovered chemistry strongly overlaps the truth.",
    "moderate_text": "Moderate overlap exists.",
    "weak_text": "Recovered chemistry differs substantially from the truth."
},

"false_discovery_rate": {
    "direction": "low",
    "excellent": 0.05,
    "good": 0.15,
    "moderate": 0.30,
    "measure": "Fraction of inferred sources that are incorrect.",
    "excellent_text": "Very few false discoveries occur.",
    "good_text": "False discoveries remain limited.",
    "moderate_text": "False discoveries are noticeable.",
    "weak_text": "False discoveries dominate the inferred solution."
},

"cost_ratio": {
    "direction": "target1",
    "excellent": 0.05,
    "good": 0.10,
    "moderate": 0.20,
    "measure": "Agreement between inferred Bayesian cost and true Bayesian cost.",
    "excellent_text": "Recovered solution closely matches the optimal Bayesian solution.",
    "good_text": "Recovered solution has similar Bayesian plausibility.",
    "moderate_text": "Recovered solution deviates somewhat from the optimum.",
    "weak_text": "Recovered solution differs substantially from the expected optimum."
},

"posterior_entropy": {
    "direction": "low",
    "excellent": 0.5,
    "good": 1.0,
    "moderate": 2.0,
    "measure": "Overall uncertainty across candidate explanations.",
    "excellent_text": "Posterior is highly concentrated around a few explanations.",
    "good_text": "Posterior shows limited ambiguity.",
    "moderate_text": "Posterior remains moderately diffuse.",
    "weak_text": "Posterior probability is spread over many explanations."
},

"effective_sample_size": {
    "direction": "low",
    "excellent": 2,
    "good": 5,
    "moderate": 10,
    "measure": "Effective number of equally probable explanations.",
    "excellent_text": "Very few explanations dominate the posterior.",
    "good_text": "Posterior concentrates on a small number of explanations.",
    "moderate_text": "Posterior remains moderately dispersed.",
    "weak_text": "Many explanations contribute similarly."
},

"top1_posterior": {
    "direction": "high",
    "excellent": 0.70,
    "good": 0.50,
    "moderate": 0.30,
    "measure": "Posterior probability assigned to the highest-ranked explanation.",
    "excellent_text": "One explanation clearly dominates.",
    "good_text": "A preferred explanation exists.",
    "moderate_text": "Several competing explanations remain.",
    "weak_text": "No dominant explanation exists."
},

"top5_posterior_mass": {
    "direction": "high",
    "excellent": 0.95,
    "good": 0.85,
    "moderate": 0.70,
    "measure": "Posterior probability contained within the five best explanations.",
    "excellent_text": "Nearly all posterior mass is concentrated in five solutions.",
    "good_text": "Most posterior mass is concentrated in five solutions.",
    "moderate_text": "Posterior remains moderately distributed.",
    "weak_text": "Posterior is widely dispersed."
},

"recoverable_fraction": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Fraction of compounds theoretically reconstructable after removing the true source.",
    "excellent_text": "Database redundancy allows nearly complete reconstruction.",
    "good_text": "Most chemistry remains reconstructable.",
    "moderate_text": "Only partial reconstruction is possible.",
    "weak_text": "Limited reconstruction is achievable."
},

"best_jaccard": {
    "direction": "low",
    "excellent": 0.10,
    "good": 0.20,
    "moderate": 0.40,
    "measure": "Similarity between database sources during LOSO analysis.",
    "excellent_text": "Database sources are highly distinct.",
    "good_text": "Only moderate redundancy exists.",
    "moderate_text": "Database contains noticeable redundancy.",
    "weak_text": "Database sources strongly overlap."
}
}

def interpret_metric(metric, value):
    info = METRIC_INFO[metric]

    if info["direction"] == "high":
        score = value
    elif info["direction"] == "low":
        score = -value
    else:
        score = -abs(value - 1)

    if info["direction"] == "target1":
        d = abs(value - 1)

        if d <= info["excellent"]:
            return "Excellent", info["excellent_text"], info["measure"]

        if d <= info["good"]:
            return "Good", info["good_text"], info["measure"]

        if d <= info["moderate"]:
            return "Moderate", info["moderate_text"], info["measure"]

        return "Weak", info["weak_text"], info["measure"]

    if score >= (info["excellent"] if info["direction"] == "high" else -info["excellent"]):
        return "Excellent", info["excellent_text"], info["measure"]

    if score >= (info["good"] if info["direction"] == "high" else -info["good"]):
        return "Good", info["good_text"], info["measure"]

    if score >= (info["moderate"] if info["direction"] == "high" else -info["moderate"]):
        return "Moderate", info["moderate_text"], info["measure"]

    return "Weak", info["weak_text"], info["measure"]

logging.info(f"Done defining validation interpretation engine | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Build master validation summary
majorsep()
logging.info(f"Building master validation summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

validation_rows = []

def add_metric(category, metric_name, metric_key, value):
    interpretation, explanation, measure = interpret_metric(metric_key, value)

    validation_rows.append({
        "Category": category,
        "Metric": metric_name,
        "Value": value,
        "Interpretation": interpretation,
        "Interpretation Explanation": explanation,
        "Metric Description": measure
    })

# Synthetic perturbation robustness
add_metric(
    "Synthetic perturbation robustness",
    "Compound precision",
    "compound_precision",
    perturb_df["precision"].mean()
)

add_metric(
    "Synthetic perturbation robustness",
    "Compound recall",
    "compound_recall",
    perturb_df["recall"].mean(),
)

# Synthetic chemistry recovery
add_metric(
    "Synthetic recovery",
    "Compound F1",
    "compound_f1",
    synth_validation_df["compound_f1"].mean()
)

add_metric(
    "Synthetic recovery",
    "Jaccard similarity",
    "jaccard",
    synth_validation_df["jaccard"].mean()
)

# Overlap sensitivity
add_metric(
    "Source ambiguity",
    "Source F1",
    "source_f1",
    overlap_validation_df["source_f1"].mean()
)

# Cost validation
add_metric(
    "Cost model",
    "Cost ratio",
    "cost_ratio",
    cost_df["cost_ratio"].mean()
)

# Beam search
add_metric(
    "Beam search",
    "Posterior entropy",
    "posterior_entropy",
    beam_validation_df["posterior_entropy"].mean()
)

add_metric(
    "Beam search",
    "Effective sample size",
    "effective_sample_size",
    beam_validation_df["effective_sample_size"].mean()
)

add_metric(
    "Beam search",
    "Top-1 posterior",
    "top1_posterior",
    beam_validation_df["top1_posterior"].mean()
)

add_metric(
    "Beam search",
    "Top-5 posterior mass",
    "top5_posterior_mass",
    beam_validation_df["top5_posterior_mass"].mean()
)

# LOSO redundancy
add_metric(
    "Database redundancy",
    "Maximum Jaccard",
    "best_jaccard",
    loso_df["best_jaccard"].mean()
)

# LOSO reconstruction
add_metric(
    "Algorithm robustness",
    "Recoverable fraction",
    "recoverable_fraction",
    loso_reconstruction_df["recoverable_fraction"].mean()
)

add_metric(
    "Algorithm robustness",
    "Compound F1",
    "compound_f1",
    loso_reconstruction_df["compound_f1"].mean()
)

master_validation_df = pd.DataFrame(validation_rows)

VAL_DIR = Path(AN_OUT_DIR / "validation_outputs")
VAL_DIR.mkdir(parents=True, exist_ok=True)
master_validation_df.to_csv(VAL_DIR / "master_validation_summary.csv", index=False)

logging.info(f"Master validation summary saved to: \n{VAL_DIR}")
minorsep()
logging.info(f"Done building master validation summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()
